# Mini LM Training

## Importing Tools

In [2]:
# ----import tools----

#utilities
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#text processing
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

#LORA Fineting
from peft import LoraConfig, get_peft_model

# Tensorflow and PyTorch
import tensorflow as tf

# CollumnTransformer and Pipeline for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#evaluation
import evaluate

# MLflow
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

#Dataset
from datasets import Dataset

## Prearing Dataset

In [3]:
# Load dataset
data = pd.read_csv("../data/processed/cleaned_phishing_email_dataset.csv")
data = Dataset.from_pandas(data)

# spilit dataset
train_test_split = data.train_test_split(test_size=0.2, seed=42)
train_set = train_test_split["train"]
val_set   = train_test_split["test"]

#check dataset
print(train_set)
print(val_set)

Dataset({
    features: ['email_text', 'is_scam'],
    num_rows: 11696
})
Dataset({
    features: ['email_text', 'is_scam'],
    num_rows: 2925
})


## Model URL

In [4]:
minilm_id = "microsoft/Multilingual-MiniLM-L12-H384"

## Tokenizing data

In [5]:
tokenizer = AutoTokenizer.from_pretrained(minilm_id)

# tokenize function
def tokenize_function(examples):
    return tokenizer(examples["email_text"], 
    truncation=True, padding="max_length", max_length=256)

# tokenized dataset
tokenized_train_set = train_set.map(tokenize_function, batched=True)
tokenized_val_set   = val_set.map(tokenize_function, batched=True)

# Rename column to 'labels' so the Trainer can find it
tokenized_train_set = tokenized_train_set.rename_column("is_scam", "labels")
tokenized_val_set = tokenized_val_set.rename_column("is_scam", "labels")


# print tokenized dataset
print(tokenized_train_set)
print(tokenized_val_set)

Map:   0%|          | 0/11696 [00:00<?, ? examples/s]

Map:   0%|          | 0/2925 [00:00<?, ? examples/s]

Dataset({
    features: ['email_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 11696
})
Dataset({
    features: ['email_text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2925
})


## Configurate Training

In [6]:
from scipy.special import softmax # Standard for NumPy-based metrics
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

# load model
minilm_model = AutoModelForSequenceClassification.from_pretrained(
    minilm_id, num_labels=2)

# LORA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.1,
    task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "dense"]
)

# apply LORA
minilm_lora = get_peft_model(minilm_model, lora_config)

# define evaluation metric recall, precision, AUC
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")
auc_metric       = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall":    recall_score(labels, preds),
    }


# training arguments
training_args = TrainingArguments(
    output_dir="./results_lora_minilm",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy="best", # save the model at the end of each epoch
    greater_is_better=True, # for recall, higher is better
    load_best_model_at_end=True, # load the best model at the end of training
    metric_for_best_model='recall'
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/Multilingual-MiniLM-L12-H384
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training and Evaluation

In [7]:
tf.random.set_seed(42)
# trainer
trainer_minilm_lora = Trainer(
    model=minilm_lora,
    args=training_args,
    train_dataset=tokenized_train_set,
    eval_dataset=tokenized_val_set,
    compute_metrics=compute_metrics,
)

# Train the model
trainer_minilm_lora.train()

# Evaluate the model
eval_results = trainer_minilm_lora.evaluate()
print(eval_results)

/opt/miniconda3/envs/dlenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
500,0.418199
1000,0.187069
1500,0.142225
2000,0.130601
2500,0.121263


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.121263,0.122765,2924,0.956581,0.956462,0.963398,0.949626


{'eval_loss': 0.1227651983499527, 'eval_accuracy': 0.9565811965811966, 'eval_f1': 0.9564621186150154, 'eval_precision': 0.9633977900552486, 'eval_recall': 0.9496255956432947}


## LOG Model and Evaluation to ML flow

In [12]:
#set mlflow experiment
mlflow.set_tracking_uri("http://localhost:8080")
mlflow.set_experiment("scam-detector-finetuning")

# log model tokenizer and metrics to mlflow
with mlflow.start_run(run_name="LORA_FineTuning_Minilm_v1") as run:
    mlflow.log_params(training_args.to_dict())
    mlflow.log_metrics(eval_results)
    mlflow.transformers.log_model(
        transformers_model= {
            "model": minilm_lora,
            "tokenizer": tokenizer
        },
        artifact_path='lora_minilm_model',
        task='text-classification'
    )

# verify if the model is logged in mlflow
client = MlflowClient()
experiment = client.get_experiment_by_name("scam-detector-finetuning")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
print(runs)

2026/05/03 14:39:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:39:38 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026/05/03 14:39:39 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository microsoft/Multilingual-MiniLM-L12-H384 will be logged instead.
2026/05/03 14:39:39 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into memory, we don't infer the pip requirement for the model. Instead, we will use the default requirements, but it may not capture all required pip libraries for the model. Consider providing the 

🏃 View run LORA_FineTuning_Minilm_v1 at: http://localhost:8080/#/experiments/1/runs/149c470308ad4d4ea886051b09346fd6
🧪 View experiment at: http://localhost:8080/#/experiments/1
[<Run: data=<RunData: metrics={'eval_accuracy': 0.9565811965811966,
 'eval_f1': 0.9564621186150154,
 'eval_loss': 0.1227651983499527,
 'eval_precision': 0.9633977900552486,
 'eval_recall': 0.9496255956432947}, params={'accelerator_config': "{'split_batches': False, 'dispatch_batches': None, "
                       "'even_batches': True, 'use_seedable_sampler': True, "
                       "'non_blocking': False, 'gradient_accumulation_kwargs': "
                       'None}',
 'adam_beta1': '0.9',
 'adam_beta2': '0.999',
 'adam_epsilon': '1e-08',
 'auto_find_batch_size': 'False',
 'average_tokens_across_devices': 'True',
 'batch_eval_metrics': 'False',
 'bf16': 'False',
 'bf16_full_eval': 'False',
 'data_seed': 'None',
 'dataloader_drop_last': 'False',
 'dataloader_num_workers': '0',
 'dataloader_persistent_